# Universe applied test

This notebook shows the simplified final structure applied directly.

Key idea:
- `Universe` is the edited main object
- it behaves like your earlier `Portfolio`
- but now it can store multiple constructions on the same loaded dataset

In [ ]:
from pathlib import Path

from portafolios import Universe, Portfolio
from portafolios.data.local_loader import local_loader
from portafolios.constructores.equal_weight import EqualWeightConstructor
from portafolios.constructores.naive import Naive
from portafolios.constructores.markowitz import Markowitz

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / "data_clean_stock_data.csv"

print("Universe is Portfolio:", Universe is Portfolio)
print("CSV exists:", CSV_PATH.exists())

## 1. Same creation pattern as before

In [ ]:
u = Universe(
    tickers=["AAPL", "MSFT", "AMZN", "GOOG"],
    start="2020-01-01",
    end="2020-06-30",
    loader=local_loader,
    loader_kwargs={"path": CSV_PATH, "prefer_adj_close": True},
    freq="B",
).preparar_datos()

u.prices.head()

In [ ]:
print("Tickers:", u.tickers)
print("Prices shape:", u.prices.shape)
print("Returns shape:", u.returns.shape)
print("Metadata:", u.metadata)

## 2. Apply multiple constructions on the same object

In [ ]:
u.construir(EqualWeightConstructor(), label="equal_weight_demo", ann_factor=252)
u.construir(Naive(), label="naive_demo", ann_factor=252)
u.construir(Markowitz(), label="markowitz_demo", ann_factor=252, ret_kind="simple", allow_short=False)

u.list_constructions()

## 3. Active construction still behaves like before

In [ ]:
print("Active label:", u.active_label)
print("Current active weights:")
u.weights

## 4. Stored constructions are now available for comparison

In [ ]:
for name in u.list_constructions():
    result = u.get_construction(name)
    print("-", result.name, "|", result.method, "| selected:", result.selected_assets)


In [ ]:
markowitz_result = u.get_construction("markowitz_demo")
print("Name:", markowitz_result.name)
print("Method:", markowitz_result.method)
print("Params:", markowitz_result.params)
print("Metrics:", markowitz_result.metrics_insample)
markowitz_result.weights

## 5. In-sample comparison table

In [ ]:
comparison = u.compare_insample_metrics()
comparison

## 6. Duplicate labels are blocked

In [ ]:
try:
    u.construir(EqualWeightConstructor(), label="equal_weight_demo")
except Exception as exc:
    print(type(exc).__name__, str(exc))